# Topic 3 — NumPy

> Standalone topic note, taught in subtopics toward the scRNA-seq count matrix. Not the project living guide. Subtopics are appended as we cover them.

---

## 3.1 — The `ndarray`

### 1. What it is

**NumPy** (Numerical Python) is the foundational numerical library of scientific Python. Its central object is the **`ndarray`** — an N-dimensional array: a grid of values, **all of the same type**, indexed by integers along each dimension.

Why it matters immediately: **the scRNA-seq count matrix *is* an `ndarray`** — a 2-D grid where rows = cells, columns = genes, and each value is how many times a gene was counted in a cell. In AnnData, the expression matrix `.X` is a NumPy array (or a sparse cousin). NumPy is the substrate the whole analysis sits on.

### 2. Why it exists — the problem with Python lists

- **Memory.** A Python list of a million numbers is a million boxed objects (each a full Python object) plus pointers, scattered in memory. An `ndarray` stores raw values in **one contiguous block**, same type, minimal overhead → far smaller.
- **Speed via vectorization.** Adding 1 to every element of a list loops in the interpreter, one slow step at a time. NumPy applies it to the whole array in **compiled C**, no Python loop → routinely 10–100× faster.
- **Scale** (the *consequence* of the two above). A real matrix can be 50,000 cells × 20,000 genes = a billion entries. Lists are hopeless; NumPy (+ sparse storage later) makes it tractable.

The one thing lists do better — mixed types, dynamic growth — is exactly what you *don't* want for a homogeneous numeric grid.

### 3. When it is used

Everywhere, mostly underneath you: **AnnData's `.X`, all of pandas, and nearly every scanpy computation are NumPy (or SciPy sparse) under the hood.** Understanding the `ndarray` demystifies every layer above it.

### 4. How it works internally

An `ndarray` = a **contiguous buffer** of raw values + **metadata** describing how to read it: `dtype` (value type), `shape` (size per dimension), and *strides* (bytes to step per axis).

Key consequence: because the array is *metadata over a shared buffer*, slicing/reshaping can return a **view** — a new wrapper pointing at the *same data* — not a copy. So **modifying a view modifies the original**. This is NumPy's most common footgun (tackled in 3.3). File away now: *slicing does not necessarily copy.*

### 5. Reading `.shape` — the cells×genes convention (and a Seurat trap)

- `.shape` is `(rows, cols)`. In **AnnData/scanpy this is always `(n_cells, n_genes)`** — rows = cells (observations), cols = genes (variables). So `.shape[0]` = cells, `.shape[1]` = genes.
- **Verify, don't just trust:** check the row count against your barcode count and the column count against your gene count. If swapped, you'll catch it instantly.
- **Transpose trap:** Seurat, Bioconductor, and the **raw 10x MTX files store it the other way — genes × cells**. The same data is transposed depending on the tool; scanpy transposes MTX to cells × genes on read. If your "cell" count is ~20,000, you're looking at a transposed matrix.

### 6. Common mistakes

- **Treating an array like a list** — arrays are fixed-size; no cheap `.append`. Appending in a loop usually means you should be vectorizing.
- **Assuming operations copy** — a slice can be a view; editing a "subset" can change the original.
- **Ignoring `dtype`** — all elements share one type. Raw counts are integers; normalization turns them into floats. Mixing causes silent surprises (e.g. integer truncation).

### 7. Best practices

- Prefer **vectorized array operations over Python loops**.
- **Stay aware of `dtype`** — know if you hold integer counts or float expression values.
- **Copy deliberately** with `.copy()` when a subset must be independent of the original (see 3.3).

### 8. Alternatives (and why NumPy)

| Option | Note |
|---|---|
| Python lists | Flexible, slow, memory-heavy → wrong for numeric grids |
| stdlib `array` | Typed but 1-D, feature-poor → rarely used |
| pandas / xarray | Richer labeled structures, but **built on** NumPy |
| SciPy sparse | Right choice when the matrix is mostly zeros (single-cell is ~90%+ zeros); a specialization met when loading real data |

Dense `ndarray` first for learning mechanics; sparse comes with real data.

### 9. Hands-on
 
Run under the **`SingleCell` kernel** (verify with `sys.executable`).

In [8]:
import numpy as np
 
# Toy "counts matrix": 3 cells (rows) x 4 genes (columns)
counts = np.array([
    [0, 3, 1, 0],
    [2, 0, 0, 5],
    [1, 1, 4, 0],
])
 
print(counts)
print("shape:", counts.shape)   # (3, 4)  -> (n_cells, n_genes)
print("ndim: ", counts.ndim)    # 2       -> two dimensions (a matrix)
print("size: ", counts.size)    # 12      -> total number of values
print("dtype:", counts.dtype)   # int64   -> every value shares ONE type

[[0 3 1 0]
 [2 0 0 5]
 [1 1 4 0]]
shape: (3, 4)
ndim:  2
size:  12
dtype: int64


- `np.array([...])` converts a list-of-lists into an `ndarray` (homogeneous, fixed-size).
- `.shape` = `(rows, cols)` = `(3, 4)`; read as `(n_cells, n_genes)`.
- `.ndim` = 2 → a matrix. `.size` = 12 → total elements. `.dtype` = `int64` → integer counts (becomes float after normalization).

Feel vectorization:

In [9]:
counts + 1        # adds 1 to EVERY element at once — compiled C, no loop

array([[1, 4, 2, 1],
       [3, 1, 1, 6],
       [2, 2, 5, 1]])

vs. the list version, which means something else entirely (and errors here):

In [12]:
py = [[0, 3, 1, 0], [2, 0, 0, 5], [1, 1, 4, 0]]
py + 1 #->  TypeError: can only concatenate list to list

TypeError: can only concatenate list (not "int") to list

That contrast is *why NumPy exists*: `+ 1` = "add one to each number" on an array; meaningless on a list.
 
### 10. Summary
 
The `ndarray` is a contiguous, single-`dtype`, fixed-size N-dimensional grid. It beats Python lists on **memory** (compact contiguous storage) and **speed** (vectorized compiled C) — which together make **scale** tractable. The count matrix is a 2-D `ndarray` shaped `(n_cells, n_genes)`; defining attributes are `.shape`, `.ndim`, `.size`, `.dtype`. Internally it's a shared buffer + metadata, so slices can be **views** (not copies) — the trap for 3.3.
 